In [ ]:
# Cell 0: GPU 確認
import subprocess, torch
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)
assert torch.cuda.is_available(), '錯誤：沒有 GPU！請檢查 Runtime 設定'
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu_name}')
print(f'VRAM: {vram_gb:.1f} GB')
if vram_gb < 70:
    print('WARNING: VRAM < 70GB，建議使用 A100 80GB')
else:
    print('✓ GPU 規格符合需求')

In [ ]:
# Cell 1: 使用者設定（必填）
# ============================================================
# 請填入你的資訊
# ============================================================

HF_USERNAME = 'Lebruhbruh'   # <-- 改成你的 HF username
HF_TOKEN    = ''   # <-- 貼上你的 Write Token

# 原始資料集（AllenAI molmoact2-bimanualyam-dataset, LeRobot v3.0, 30fps,
# bi_yam_follower，共 284 episodes、單一任務「charging」四階段）
SOURCE_REPO_IDS = [
    'allenai/19012026-charging-01',
    'allenai/19012026-charging-02',
    'allenai/19012026-charging-03',
    'allenai/19012026-charging-04',
    'allenai/19012026-charging-05',
    'allenai/19012026-charging-06',
]

# 六個 dataset 合併後的新 repo（Cell 4b 會建立並上傳）
MERGED_DATASET = f'{HF_USERNAME}/charging_bimanual_merged'

# 後續 cell 都以「合併後的 dataset」作為 source
SOURCE_DATASET = MERGED_DATASET

# 標注後資料集 repo（會自動建立）
ANNOTATED_REPO_ID = f'{HF_USERNAME}/charging_bimanual_annotated'

# 訓練好的 SARM 模型 repo
MODEL_REPO_ID = f'{HF_USERNAME}/sarm-charging-bimanual'

# 合併後 dataset 只有 1 個任務（charging，四階段）
TASK_INDEX = 0

print(f'來源資料集 ({len(SOURCE_REPO_IDS)} 個):')
for r in SOURCE_REPO_IDS:
    print(f'  - {r}')
print(f'合併後資料集:   {MERGED_DATASET}')
print(f'標注資料集:     {ANNOTATED_REPO_ID}')
print(f'SARM 模型:      {MODEL_REPO_ID}')


In [ ]:
# Cell 1b: 進度回饋工具（後續長時 cell 共用）
# ============================================================
# progress_stage(title, steps, heartbeat_seconds): context manager
#   - 開頭印 ━━━ 標題 ━━━ 與步驟清單
#   - 背景 thread 每 N 秒印一次 ⏱ 心跳（含已經過時間）
#   - 結尾印 ✓ 與總耗時
# 預期被 Cell 2 / 4b / 9a / 10 / 12 / 13 / 14 等長時 cell 使用。
# ============================================================
import threading, time, contextlib
from datetime import timedelta

def _fmt_elapsed(seconds: float) -> str:
    return str(timedelta(seconds=int(seconds)))

@contextlib.contextmanager
def progress_stage(title: str, steps=None, heartbeat_seconds: int = 30):
    print(f'\n━━━ {title} ━━━')
    if steps:
        for i, step in enumerate(steps, 1):
            print(f'  {i}) {step}')
        print()

    stop_evt = threading.Event()
    start = time.monotonic()

    def _heartbeat():
        while not stop_evt.wait(heartbeat_seconds):
            print(f'  ⏱  已執行 {_fmt_elapsed(time.monotonic() - start)}', flush=True)

    th = threading.Thread(target=_heartbeat, daemon=True)
    th.start()
    error = None
    try:
        yield
    except BaseException as e:
        error = e
        raise
    finally:
        stop_evt.set()
        th.join(timeout=1)
        total = _fmt_elapsed(time.monotonic() - start)
        if error is None:
            print(f'\n✓ {title} 完成，總耗時 {total}')
        else:
            print(f'\n✗ {title} 失敗（{type(error).__name__}），已執行 {total}')

print('✓ progress_stage 已就緒')


In [ ]:
# Cell 2: 安裝 LeRobot + SARM
import subprocess, sys

with progress_stage(
    '安裝 LeRobot + SARM（預計 2-5 分鐘）',
    steps=['pip install lerobot[sarm] from GitHub', '確認 lerobot / transformers 版本'],
    heartbeat_seconds=30,
):
    print('1/2 安裝 LeRobot + SARM...')
    # CLAUDE.md 規則 4：>30 秒指令不可 capture_output；讓 pip 即時輸出
    r = subprocess.run(
        'pip install -q "git+https://github.com/huggingface/lerobot.git#egg=lerobot[sarm]"',
        shell=True, text=True,
    )
    if r.returncode != 0:
        raise RuntimeError(f'pip install 失敗，返回碼 {r.returncode}')
    print('  ✓ LeRobot 安裝完成')

    print('2/2 確認安裝...')
    check_cmd = 'import lerobot, transformers; print(lerobot.__version__); print(transformers.__version__)'
    r = subprocess.run([sys.executable, '-c', check_cmd], capture_output=True, text=True)
    if r.returncode == 0:
        versions = r.stdout.strip().split()
        print(f'  LeRobot: {versions[0]}')
        print(f'  Transformers: {versions[1] if len(versions)>1 else "?"}')
    else:
        raise RuntimeError(f'版本確認失敗: {r.stderr}')


In [ ]:
# Cell 3: 修補 Transformers 5.x Bug（CRITICAL）
import os, glob, torch, subprocess
import lerobot

lerobot_root = os.path.dirname(lerobot.__file__)
print(f"LeRobot 安裝路徑: {lerobot_root}")

hits = glob.glob(os.path.join(lerobot_root, "**", "processor_sarm.py"), recursive=True)
if not hits:
    print("lerobot_root 下找不到，往上一層搜尋...")
    hits = glob.glob(os.path.join(os.path.dirname(lerobot_root), "**", "processor_sarm.py"), recursive=True)

if not hits:
    raise FileNotFoundError(
        "找不到 processor_sarm.py，請確認 lerobot[sarm] 安裝成功。"
        "可在新 cell 執行: !find /usr/local/lib -name processor_sarm.py"
    )

processor_path = hits[0]
print(f"修補目標: {processor_path}")

with open(processor_path) as f:
    content = f.read()

patched = False

indent = "\n        "
old_img = "embeddings = self.clip_model.get_image_features(**inputs).detach().cpu()"
new_img = indent.join([
    "output = self.clip_model.get_image_features(**inputs)",
    "if not isinstance(output, torch.Tensor):",
    "    output = output.pooler_output",
    "embeddings = output.detach().cpu()",
])
if old_img in content:
    content = content.replace(old_img, new_img)
    print("✓ 已修補 image features")
    patched = True
else:
    print("ℹ image features 不需修補")

old_txt = "embeddings = self.clip_model.get_text_features(**inputs).detach().cpu()"
new_txt = indent.join([
    "output = self.clip_model.get_text_features(**inputs)",
    "if not isinstance(output, torch.Tensor):",
    "    output = output.pooler_output",
    "embeddings = output.detach().cpu()",
])
if old_txt in content:
    content = content.replace(old_txt, new_txt)
    print("✓ 已修補 text features")
    patched = True
else:
    print("ℹ text features 不需修補")

with open(processor_path, "w") as f:
    f.write(content)

if patched:
    r = subprocess.run(["grep", "-n", "pooler_output", processor_path],
                       capture_output=True, text=True)
    print("\n驗證修補（應看到 pooler_output）:")
    print(r.stdout)
else:
    print("\n✓ 程式碼已是最新版，無需修補")


In [ ]:
# Cell 4: HuggingFace 登入
from huggingface_hub import login
login(token=HF_TOKEN)
print('✓ HuggingFace 登入成功')

In [ ]:
# Cell 4b: 合併 6 個 charging dataset 到單一 HF repo
# ============================================================
# 1) 對每個來源 repo 構造 LeRobotDatasetMetadata 拿到 revision，再用
#    snapshot_download 下載完整檔案（含 videos）到同一個 snapshot 資料夾。
# 2) Monkey-patch pd.read_parquet（v6）：用 pyarrow.parquet.read_table 直讀；
#    若 to_pandas() 因 parquet 內嵌 ArrowDtype metadata 而炸，retry with
#    ignore_metadata=True 跳過 metadata 解析。
# 3) 呼叫 aggregate_datasets() 合併到 /content/merged_dataset。
# 4) 上傳合併結果並建立 v3.0 git tag（LeRobotDataset 預設用 v3.0 tag 當 revision，
#    沒有 tag 後續 LeRobotDataset(MERGED_DATASET) 會炸——見 debug.md E15）。
# 5) 驗證合併結果（用 local root 跳過 Hub tag propagation）。
# 若 MERGED_DATASET 已存在於 Hub（且有 meta/info.json）則直接跳過合併。
# ============================================================
import os, shutil
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq
from huggingface_hub import HfApi, snapshot_download
from lerobot.datasets.aggregate import aggregate_datasets
from lerobot.datasets.dataset_metadata import LeRobotDatasetMetadata
from lerobot.utils.constants import HF_LEROBOT_HUB_CACHE

api = HfApi(token=HF_TOKEN)

def _merged_is_complete():
    """Hub repo 存在 + 含 meta/info.json 才算合併好（防 ghost repo / E14）。"""
    try:
        api.dataset_info(MERGED_DATASET)
    except Exception:
        return False
    try:
        api.hf_hub_download(MERGED_DATASET, 'meta/info.json',
                            repo_type='dataset', token=HF_TOKEN)
        return True
    except Exception:
        return False

aggr_root = Path('/content/merged_dataset')

if _merged_is_complete():
    print(f'✓ MERGED_DATASET 已完整存在於 Hub，跳過合併: {MERGED_DATASET}')
else:
    with progress_stage(
        f'合併 {len(SOURCE_REPO_IDS)} 個 charging dataset',
        steps=[
            f'解析每個 repo 的 LeRobot revision（CODEBASE_VERSION="v3.0"）',
            f'下載完整檔案（含 videos）到 {HF_LEROBOT_HUB_CACHE}',
            'Bypass pd.read_parquet（v6：to_pandas 失敗則 retry ignore_metadata=True）',
            '呼叫 aggregate_datasets() 進行合併',
            f'建立 HF repo、upload 並打 v3.0 tag → {MERGED_DATASET}',
        ],
        heartbeat_seconds=30,
    ):
        # ---- 1: 解析 revision 後完整下載 source repos
        for i, repo in enumerate(SOURCE_REPO_IDS, 1):
            print(f'  [{i}/{len(SOURCE_REPO_IDS)}] resolving revision for {repo}...')
            meta = LeRobotDatasetMetadata(repo)
            rev = meta.revision
            print(f'    revision={rev}; snapshot_download 完整檔案中...')
            snapshot_download(
                repo_id=repo,
                repo_type='dataset',
                revision=rev,
                cache_dir=HF_LEROBOT_HUB_CACHE,
                token=HF_TOKEN,
            )

        # ---- 2: monkey-patch pd.read_parquet → pyarrow 直讀 + ignore_metadata fallback
        _orig_read_parquet = pd.read_parquet
        _patch_stats = {'total': 0, 'ignore_meta_fallbacks': 0, 'fallback_files': []}

        def _read_parquet_via_pyarrow(*args, **kwargs):
            _patch_stats['total'] += 1
            if args:
                path = args[0]
            else:
                path = kwargs.get('path') or kwargs.get('path_or_buf')
            columns = kwargs.get('columns', None)
            table = pq.read_table(str(path), columns=columns)
            try:
                df = table.to_pandas()
            except TypeError as e:
                if '[pyarrow]' in str(e) or 'not understood' in str(e):
                    df = table.to_pandas(ignore_metadata=True)
                    _patch_stats['ignore_meta_fallbacks'] += 1
                    fname = str(path).split('/')[-1]
                    if fname not in _patch_stats['fallback_files']:
                        _patch_stats['fallback_files'].append(fname)
                else:
                    raise
            for col in df.columns:
                if hasattr(df[col].dtype, 'pyarrow_dtype'):
                    df[col] = list(df[col])
            if hasattr(df.index.dtype, 'pyarrow_dtype'):
                df.index = pd.Index(list(df.index), name=df.index.name)
            return df

        pd.read_parquet = _read_parquet_via_pyarrow
        print('  pd.read_parquet 已 patch（v6: pyarrow direct + ignore_metadata fallback）')

        # ---- 3: aggregate
        if aggr_root.exists():
            shutil.rmtree(aggr_root)

        try:
            aggregate_datasets(
                repo_ids=SOURCE_REPO_IDS,
                aggr_repo_id=MERGED_DATASET,
                aggr_root=aggr_root,
            )
        finally:
            pd.read_parquet = _orig_read_parquet
            print(f'  pd.read_parquet 已復原；patch 共呼叫 {_patch_stats["total"]} 次，'
                  f'其中 {_patch_stats["ignore_meta_fallbacks"]} 次走 ignore_metadata fallback')
            if _patch_stats['fallback_files']:
                print(f'  觸發 fallback 的檔案類型: {_patch_stats["fallback_files"][:5]}')

        # ---- 4: 上傳合併結果 + 建立 v3.0 tag
        api.create_repo(repo_id=MERGED_DATASET, repo_type='dataset', exist_ok=True)
        api.upload_folder(
            folder_path=str(aggr_root),
            repo_id=MERGED_DATASET,
            repo_type='dataset',
        )
        print(f'✓ 已上傳合併 dataset: https://huggingface.co/datasets/{MERGED_DATASET}')

        # LeRobotDatasetMetadata 預設用 revision="v3.0"（CODEBASE_VERSION）→ 找這個 tag。
        # 沒打 tag 後續所有 LeRobotDataset(MERGED_DATASET) 會炸（見 debug.md E15）
        try:
            api.create_tag(
                repo_id=MERGED_DATASET, repo_type='dataset',
                tag='v3.0', exist_ok=True,
            )
            print('✓ 已建立 v3.0 git tag')
        except Exception as e:
            print(f'⚠ 建立 v3.0 tag 失敗（合併本身已成功）: {e}')
            print('  可在 Hub 網頁手動建：Settings → Tags → New Tag → v3.0 (from main)')

# ---- 5: 確認合併結果（local 優先；local 沒有再從 Hub 拉）
print('\n── 合併結果驗證 ──')
local_meta_info = aggr_root / 'meta' / 'info.json'
try:
    if local_meta_info.exists():
        meta = LeRobotDatasetMetadata(MERGED_DATASET, root=aggr_root)
        src_label = 'local /content/merged_dataset'
    else:
        meta = LeRobotDatasetMetadata(MERGED_DATASET)
        src_label = 'Hub'
    print(f'  資料來源:       {src_label}')
    print(f'  total_episodes: {meta.total_episodes}')
    print(f'  total_frames:   {meta.total_frames}')
    print(f'  fps:            {meta.fps}')
    print(f'  robot_type:     {meta.robot_type}')
    print(f'  total_tasks:    {len(meta.tasks)}')
except Exception as e:
    print(f'⚠ 驗證讀取失敗（不一定代表合併有問題）: {type(e).__name__}: {e}')
    print(f'  請到 https://huggingface.co/datasets/{MERGED_DATASET} 直接看 info.json')


In [ ]:
# Cell 5: 資料集預覽
from lerobot.datasets.lerobot_dataset import LeRobotDataset

print("載入合併後 charging dataset metadata（不下載影片）...")
dataset = LeRobotDataset(SOURCE_DATASET)

print(f"Episodes 總數: {dataset.num_episodes}")
print(f"Frames 總數:   {dataset.num_frames:,}")
print(f"FPS:           {dataset.fps}")
print(f"格式版本:      {dataset.meta.info.get('codebase_version', 'unknown')}")

print("\n可用 Feature Keys:")
for k in dataset.features:
    print(f"  {k}")

image_keys = [k for k in dataset.features if "images" in k]
print(f"\n攝影機 Keys: {image_keys}")

# meta.tasks 可能是 dict、list 或 numpy array，統一用 enumerate 處理
print("\n任務描述 (task_index → description):")
tasks = dataset.meta.tasks
if hasattr(tasks, "items"):
    for idx, task in tasks.items():
        print(f"  [{idx}] {task}")
else:
    for idx, task in enumerate(tasks):
        print(f"  [{idx}] {task}")


In [ ]:
# Cell 5b: 全部 episode 列表
# ============================================================
# 合併後的 charging dataset 是「單一任務（四階段）」，所有 episode
# 都屬於同一個任務，因此不需要 task_index 過濾——直接使用全部 episodes。
# ============================================================
all_task_episodes = list(range(dataset.num_episodes))

# 顯示 meta 裡的任務描述（合併後通常為 1 條，若 6 個 source 的 task 文字
# 完全相同會被去重；不同則會保留 N 條）
print('合併後任務列表 (task_index → description):')
tasks = dataset.meta.tasks
if hasattr(tasks, 'items'):
    for idx, task in tasks.items():
        print(f'  [{idx}] {task}')
else:
    for idx, task in enumerate(tasks):
        print(f'  [{idx}] {task}')

# 使用全部 episodes（284 個）
TASK_EPISODES = all_task_episodes
print(f'\n總共 {len(TASK_EPISODES)} 個 episodes 將被標注')
print(f'  前 10 個 episode indices: {TASK_EPISODES[:10]}')


In [ ]:
# Cell 6: 觀看範例 Episodes（確認子任務名稱）
from lerobot.datasets.lerobot_dataset import LeRobotDataset
import matplotlib.pyplot as plt
import numpy as np

PREVIEW_VIDEO_KEY = 'observation.images.top'

preview_eps = TASK_EPISODES[:3]
print(f'載入 episodes {preview_eps} 的 {PREVIEW_VIDEO_KEY} 影像...')
ds_preview = LeRobotDataset(SOURCE_DATASET, episodes=preview_eps)

fig, axes = plt.subplots(3, 5, figsize=(20, 12))
for row, ep_idx in enumerate(preview_eps):
    ep_frames = [i for i, s in enumerate(ds_preview)
                 if s['episode_index'].item() == ep_idx]
    sample_indices = np.linspace(0, len(ep_frames)-1, 5, dtype=int)
    for col, fi in enumerate(sample_indices):
        frame = ds_preview[ep_frames[fi]]
        img = frame[PREVIEW_VIDEO_KEY].numpy()
        if img.max() <= 1.0:
            img = (img * 255).astype(np.uint8)
        axes[row][col].imshow(img.transpose(1, 2, 0) if img.ndim == 3 else img)
        axes[row][col].set_title(f'Ep {ep_idx}, t={fi}')
        axes[row][col].axis('off')

plt.suptitle('Charging Task Episodes 預覽（top camera）', fontsize=14)
plt.tight_layout()
plt.savefig('/content/episode_preview.png', dpi=100, bbox_inches='tight')
plt.show()
print('✓ 預覽圖已儲存至 /content/episode_preview.png')


In [ ]:
# Cell 7: 設定標注參數（根據 Cell 6 觀察調整）
# ============================================================
# Charging 任務四階段：
#   1) 抓取手機
#   2) 翻轉手機側放
#   3) 抓充電線並插入手機
#   4) 打開延長線電源
# ============================================================

DENSE_SUBTASKS = (
    'Pick up the phone,'
    'Flip the phone sideways,'
    'Pick up the charging cable and plug it into the phone,'
    'Turn on the power of the extension cord'
)

VIDEO_KEY = 'observation.images.top'
VLM_MODEL = 'Qwen/Qwen3-VL-30B-A3B-Instruct'

# 前 10 個 episodes 做測試
TEST_EPISODES = TASK_EPISODES[:10]
RUN_ALL_EPISODES = False   # 確認標注正確後改為 True

print(f'子任務: {DENSE_SUBTASKS}')
print(f'攝影機: {VIDEO_KEY}')
print(f'VLM:    {VLM_MODEL}')
print(f'測試 episodes: {TEST_EPISODES}')
print(f'全量 episodes: {len(TASK_EPISODES)} 個')


In [ ]:
# Cell 7b: AV1 codec 診斷 + patch extract_frame 改用 pyav
# ============================================================
# charging dataset 的影片是 AV1 編碼。Colab 預設 cv2 可能不帶 AV1 decoder，
# 導致 lerobot 的 subtask_annotation.extract_frame() 全部回傳 None，
# Cell 9b / Cell 11 視覺化每張圖上都會看到 "N/A"（matplotlib 的占位文字）。
#
# 這個 cell 自動偵測：
#   - 若 cv2 已能讀 AV1 → 不做任何事
#   - 若不能 → 把 lerobot 安裝目錄下的 subtask_annotation.py 的 extract_frame
#     函式體改成「先試 pyav、失敗 fallback 到 cv2」。改完 reload module，
#     並驗證 patch 後能成功取到 frame。
# ============================================================
import os, glob, cv2

# ---- 1) 找一個 AV1 mp4 來測 cv2 ----
test_paths = (
    sorted(glob.glob(
        '/root/.cache/huggingface/lerobot/hub/'
        'datasets--allenai--19012026-charging-01/snapshots/*/'
        'videos/observation.images.top/chunk-000/*.mp4'
    ))
    or sorted(glob.glob(
        '/content/merged_dataset/videos/observation.images.top/chunk-000/*.mp4'
    ))
)

cv2_can_read_av1 = False
if test_paths:
    cap = cv2.VideoCapture(test_paths[0])
    if cap.isOpened():
        ret, frame = cap.read()
        cv2_can_read_av1 = bool(ret and frame is not None)
    cap.release()
    print(f'AV1 codec 測試 ({os.path.basename(test_paths[0])}): '
          f'cv2 讀取 {"OK" if cv2_can_read_av1 else "FAIL"}')
else:
    print('⚠ 找不到 AV1 測試 mp4（先跑過 Cell 4b 確保 dataset 已合併/下載）')

# ---- 2) cv2 不行 → patch lerobot 的 extract_frame ----
if cv2_can_read_av1:
    print('✓ cv2 已支援 AV1，不需要 patch')
else:
    import lerobot
    lerobot_root = os.path.dirname(lerobot.__file__)
    hits = glob.glob(
        os.path.join(lerobot_root, '**', 'subtask_annotation.py'),
        recursive=True,
    )
    if not hits:
        hits = glob.glob(
            os.path.join(os.path.dirname(lerobot_root), '**', 'subtask_annotation.py'),
            recursive=True,
        )
    if not hits:
        raise FileNotFoundError(
            '找不到 subtask_annotation.py，請確認 lerobot[sarm] 安裝成功'
        )

    path = hits[0]
    print(f'修補目標: {path}')

    with open(path) as f:
        content = f.read()

    old_body = (
        'def extract_frame(video_path: Path, timestamp: float) -> np.ndarray | None:\n'
        '    """Extract a single frame from video at given timestamp."""\n'
        '    cap = cv2.VideoCapture(str(video_path))\n'
        '    if not cap.isOpened():\n'
        '        return None\n'
        '    cap.set(cv2.CAP_PROP_POS_MSEC, timestamp * 1000)\n'
        '    ret, frame = cap.read()\n'
        '    cap.release()\n'
        '    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB) if ret else None'
    )

    new_body = (
        'def extract_frame(video_path: Path, timestamp: float) -> np.ndarray | None:\n'
        '    """Extract a single frame (patched: pyav-first, cv2-fallback)."""\n'
        '    try:\n'
        '        import av\n'
        '        with av.open(str(video_path)) as container:\n'
        '            stream = container.streams.video[0]\n'
        '            target_pts = int(timestamp / float(stream.time_base))\n'
        '            container.seek(target_pts, any_frame=False, backward=True, stream=stream)\n'
        '            last = None\n'
        '            for frame in container.decode(stream):\n'
        '                last = frame\n'
        '                if frame.pts is None:\n'
        '                    continue\n'
        '                if float(frame.pts * stream.time_base) >= timestamp:\n'
        '                    return frame.to_ndarray(format="rgb24")\n'
        '            return last.to_ndarray(format="rgb24") if last is not None else None\n'
        '    except Exception:\n'
        '        cap = cv2.VideoCapture(str(video_path))\n'
        '        if not cap.isOpened():\n'
        '            return None\n'
        '        cap.set(cv2.CAP_PROP_POS_MSEC, timestamp * 1000)\n'
        '        ret, frame = cap.read()\n'
        '        cap.release()\n'
        '        return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB) if ret else None'
    )

    if old_body in content:
        content = content.replace(old_body, new_body)
        with open(path, 'w') as f:
            f.write(content)
        print('✓ extract_frame 已 patch → pyav-first + cv2-fallback')
    elif 'pyav-first' in content:
        print('ℹ extract_frame 已是 patch 後版本，跳過')
    else:
        print('⚠ 找不到原始 extract_frame 文本（lerobot 版本可能不同）')
        print(f'  請手動編輯 {path}，把 extract_frame 函式體改成上述 pyav 版本')

    # ---- 3) reload 並驗證能讀到 frame ----
    if test_paths:
        import importlib, sys
        mod_name = 'lerobot.data_processing.sarm_annotations.subtask_annotation'
        if mod_name in sys.modules:
            importlib.reload(sys.modules[mod_name])
        from lerobot.data_processing.sarm_annotations.subtask_annotation import extract_frame
        frame = extract_frame(test_paths[0], 0.5)
        if frame is not None:
            print(f'✓ 驗證 OK：patch 後成功取到 frame shape={frame.shape}')
        else:
            print('⚠ 驗證失敗：patch 後仍取不到 frame，請貼此 cell 完整輸出')


In [ ]:
# Cell 8: 建立標注輸出 Repo
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)
try:
    api.create_repo(repo_id=ANNOTATED_REPO_ID, repo_type='dataset', exist_ok=True)
    print(f'✓ Repo 建立成功: https://huggingface.co/datasets/{ANNOTATED_REPO_ID}')
except Exception as e:
    print(f'✗ 建立 repo 失敗: {e}'); raise

In [ ]:
# Cell 9a: 測試標注（前 10 個 charging episodes）
import subprocess, os, sys

ANNOTATE_MODULE = 'lerobot.data_processing.sarm_annotations.subtask_annotation'

cmd = [
    sys.executable, '-m', ANNOTATE_MODULE,
    '--repo-id', SOURCE_DATASET,
    '--output-repo-id', ANNOTATED_REPO_ID,
    '--dense-only',
    '--dense-subtasks', DENSE_SUBTASKS,
    '--video-key', VIDEO_KEY,
    '--model', VLM_MODEL,
    '--num-workers', '1',
    '--episodes', *[str(e) for e in TEST_EPISODES],
    '--num-visualizations', '5',
    '--output-dir', '/content/annotation_viz_test',
    '--push-to-hub',
]

print('命令:', ' '.join(cmd))

env = os.environ.copy()
env.update({'HF_TOKEN': HF_TOKEN, 'HUGGING_FACE_HUB_TOKEN': HF_TOKEN})

with progress_stage(
    f'測試標注（{len(TEST_EPISODES)} episodes，預計 20-40 分鐘）',
    steps=[
        '下載 Qwen3-VL-30B（首次 ~60GB）',
        f'逐個 episode 呼叫 VLM 標注 4 個子任務（共 {len(TEST_EPISODES)} 個）',
        '產生視覺化 PNG 到 /content/annotation_viz_test',
        '推送標注結果到 HF Hub',
    ],
    heartbeat_seconds=30,
):
    # 捕捉 stderr 以便診斷錯誤；stdout 直接顯示在 cell
    result = subprocess.run(cmd, env=env, text=True,
                            stdout=None,
                            stderr=subprocess.PIPE)

if result.returncode == 0:
    print('\n✓ 測試標注完成！請執行 Cell 9b 檢視結果')
else:
    print(f'\n✗ 標注失敗，返回碼: {result.returncode}')
    print('\n=== 錯誤訊息（最後 3000 字） ===')
    print(result.stderr[-3000:] if result.stderr else '（無 stderr 輸出）')


In [ ]:
# Cell 9b: 驗證測試標注結果
import glob, os
from IPython.display import Image as IPyImage, display

viz_files = sorted(glob.glob('/content/annotation_viz_test/*.png'))
if not viz_files:
    print('找不到視覺化圖片，請確認 Cell 9a 成功執行')
else:
    print(f'找到 {len(viz_files)} 張視覺化圖片:\n')
    for path in viz_files[:5]:
        print(f'  {os.path.basename(path)}')
        display(IPyImage(path))
    print('\n關鍵確認：')
    print('  ✓ 每個 episode 應有 4 個顏色區段（對應 4 個子任務）')
    print('  ✓ 邊界應與影片中的動作轉換點吻合')
    print('  ✗ 若所有幀都是同色 → 調整 DENSE_SUBTASKS 後重跑 Cell 9a')

In [ ]:
# Cell 10: 全量標注（確認 Cell 9b 正確後執行）
# 預計 1.5-3 小時；Colab 斷線後重跑此 Cell 即可（--skip-existing 自動續跑）
import subprocess, os, sys

ANNOTATE_MODULE = 'lerobot.data_processing.sarm_annotations.subtask_annotation'

cmd = [
    sys.executable, '-m', ANNOTATE_MODULE,
    '--repo-id', SOURCE_DATASET,
    '--output-repo-id', ANNOTATED_REPO_ID,
    '--dense-only',
    '--dense-subtasks', DENSE_SUBTASKS,
    '--video-key', VIDEO_KEY,
    '--model', VLM_MODEL,
    '--num-workers', '1',
    '--episodes', *[str(e) for e in TASK_EPISODES],
    '--skip-existing',
    '--num-visualizations', '5',
    '--output-dir', '/content/annotation_viz',
    '--push-to-hub',
]

print('⚠️  預計 1.5-3 小時，建議隔夜執行')
print('   Colab 斷線後重新執行此 Cell 即可繼續（--skip-existing）')
print('命令:', ' '.join(cmd))

env = os.environ.copy()
env.update({'HF_TOKEN': HF_TOKEN, 'HUGGING_FACE_HUB_TOKEN': HF_TOKEN})

with progress_stage(
    f'全量標注（{len(TASK_EPISODES)} episodes，預計 1.5-3 小時）',
    steps=[
        '若有現存標注 → 跳過（--skip-existing）',
        f'逐個 episode 呼叫 VLM 標注 4 個子任務（共 {len(TASK_EPISODES)} 個）',
        '產生視覺化 PNG 到 /content/annotation_viz',
        '推送標注結果到 HF Hub',
    ],
    heartbeat_seconds=30,
):
    result = subprocess.run(cmd, env=env, text=True)

if result.returncode == 0:
    print('\n✓ 全量標注完成！')
else:
    print(f'\n✗ 標注中斷（返回碼: {result.returncode}）')
    print('重新執行此 Cell 可從斷點繼續')


In [ ]:
# Cell 11: 驗證完整標注
import glob, os
from IPython.display import Image as IPyImage, display

print('最終標注視覺化:')
viz_files = sorted(glob.glob('/content/annotation_viz/*.png'))
for path in viz_files[:5]:
    print(f'\n{os.path.basename(path)}:')
    display(IPyImage(path))

print(f'\n標注資料集位置: https://huggingface.co/datasets/{ANNOTATED_REPO_ID}')

In [ ]:
# Cell 12: 訓練 SARM Reward Model（預計 45-90 分鐘）
import subprocess, os, sys, shutil

lerobot_train = shutil.which('lerobot-train') or [
    sys.executable, '-m', 'lerobot.scripts.lerobot_train'
]
if isinstance(lerobot_train, str):
    lerobot_train = [lerobot_train]

cmd = [
    *lerobot_train,
    f'--dataset.repo_id={ANNOTATED_REPO_ID}',
    '--policy.type=sarm',
    '--policy.annotation_mode=dense_only',
    f'--policy.image_key={VIDEO_KEY}',
    '--policy.state_key=observation.state',
    '--policy.n_obs_steps=8',
    '--policy.frame_gap=30',   # charging dataset @ 30fps，gap=30 covers 1 second
    '--output_dir=outputs/train/sarm_charging_bimanual',
    '--batch_size=64',
    '--steps=5000',
    f'--policy.repo_id={MODEL_REPO_ID}',
    '--wandb.enable=false',
]

print('命令:', ' '.join(cmd))

env = os.environ.copy()
env.update({'HF_TOKEN': HF_TOKEN, 'HUGGING_FACE_HUB_TOKEN': HF_TOKEN})

with progress_stage(
    '訓練 SARM Reward Model（5000 steps，預計 45-90 分鐘）',
    steps=[
        '載入 ANNOTATED_REPO_ID 的標注資料',
        '初始化 SARM policy（CLIP backbone + dense head）',
        '訓練 5000 steps（lerobot-train 內部會印 step 進度）',
        '推送模型到 HF Hub',
    ],
    heartbeat_seconds=30,
):
    result = subprocess.run(cmd, env=env, text=True)

if result.returncode == 0:
    print(f'\n✓ 訓練完成！模型已上傳至: https://huggingface.co/{MODEL_REPO_ID}')
else:
    print(f'\n✗ 訓練失敗（返回碼: {result.returncode}）')


In [ ]:
# Cell 13: 備份 Checkpoint 到 Google Drive（選擇性）
from google.colab import drive
import shutil, os

drive.mount('/content/drive')

backup_path = '/content/drive/MyDrive/sarm_checkpoints/sarm_charging_bimanual'
src = 'outputs/train/sarm_charging_bimanual'

if not os.path.exists(src):
    print('找不到 checkpoint 目錄，請確認訓練已完成')
else:
    os.makedirs(backup_path, exist_ok=True)
    with progress_stage(
        '備份 Checkpoint 到 Google Drive',
        steps=[
            f'來源: {src}',
            f'目標: {backup_path}',
            '使用 shutil.copytree 複製（大型 checkpoint 可能需數分鐘）',
        ],
        heartbeat_seconds=30,
    ):
        shutil.copytree(src, backup_path, dirs_exist_ok=True)
    print(f'✓ Checkpoint 已備份至: {backup_path}')


In [ ]:
# Cell 14: 視覺化預測結果
import subprocess, os, sys

cmd = [
    sys.executable, '-m', 'lerobot.policies.sarm.compute_rabc_weights',
    '--dataset-repo-id', ANNOTATED_REPO_ID,
    '--reward-model-path', MODEL_REPO_ID,
    '--visualize-only',
    '--num-visualizations', '5',
    '--head-mode', 'dense',
    '--output-dir', '/content/sarm_predictions_viz',
]

env = os.environ.copy()
env.update({'HF_TOKEN': HF_TOKEN, 'HUGGING_FACE_HUB_TOKEN': HF_TOKEN})

with progress_stage(
    '產生 SARM 預測視覺化',
    steps=[
        '載入訓練好的 SARM 模型',
        '對 5 個 episode 算 dense reward',
        '輸出 PNG 到 /content/sarm_predictions_viz',
    ],
    heartbeat_seconds=30,
):
    result = subprocess.run(cmd, env=env, text=True)

if result.returncode != 0:
    print(f'視覺化失敗（返回碼: {result.returncode}）')


In [ ]:
# Cell 14b: 顯示預測結果
import glob
from IPython.display import Image as IPyImage, display

print('SARM Reward Model 預測結果:')
pred_files = sorted(glob.glob('/content/sarm_predictions_viz/*.png'))
if not pred_files:
    print('找不到預測圖片，請確認 Cell 14 成功執行')
else:
    for path in pred_files[:5]:
        print(f'\n{__import__("os").path.basename(path)}:')
        display(IPyImage(path))
    print('\n預期結果：每個 episode 的進度曲線應單調遞增（0 → 1）')
    print('         子任務轉換處應有明顯的斜率變化')

In [ ]:
# Cell 15: 確認模型已上傳 HuggingFace Hub
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)
try:
    info = api.model_info(MODEL_REPO_ID)
    print(f'✓ 模型已成功上傳！')
    print(f'  名稱: {info.id}')
    print(f'  最後更新: {info.last_modified}')
    print(f'  連結: https://huggingface.co/{MODEL_REPO_ID}')
except Exception as e:
    print(f'✗ 找不到模型: {e}')
    print('  請確認 Cell 12 的訓練步驟已成功完成')